In [19]:
import cv2
import numpy as np
import mediapipe as mp
import onnxruntime as ort

from win10toast import ToastNotifier

# 初始化提醒工具
toaster = ToastNotifier()
anomaly_counter = 0    # 异常帧计数器
ALERT_LIMIT = 60       # 连续检测到 60 帧异常则报警
threshold = 0.7        # 置信度阈值

In [20]:
# --- 1. 配置 ---
actions = ['attention', 'phone', 'daydream', 'sleepy']
# 加载 ONNX 模型
VERSION = "V4.5_"
ort_sess = ort.InferenceSession(f"{VERSION}student_model.onnx")

In [21]:
from mediapipe.python.solutions.holistic import Holistic
holistic = Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5)

In [22]:
def extract_keypoints(results):
    pose = np.array([[res.x, res.y, res.z, res.visibility] for res in results.pose_landmarks.landmark]).flatten() if results.pose_landmarks else np.zeros(33*4)
    lh = np.array([[res.x, res.y, res.z] for res in results.left_hand_landmarks.landmark]).flatten() if results.left_hand_landmarks else np.zeros(21*3)
    rh = np.array([[res.x, res.y, res.z] for res in results.right_hand_landmarks.landmark]).flatten() if results.right_hand_landmarks else np.zeros(21*3)
    face = np.array([[res.x, res.y, res.z] for res in results.face_landmarks.landmark]).flatten() if results.face_landmarks else np.zeros(468*3)
    # return np.concatenate([pose, lh, rh, face]).astype(np.float32) # ONNX 必须用 float32
    return np.concatenate([pose, lh, rh])

In [23]:
def draw_alert_ui(image, action, conf, is_anomaly):
    h, w, _ = image.shape
    if is_anomaly:
        # 红色全屏边框
        cv2.rectangle(image, (0, 0), (w, h), (0, 0, 255), 15)
        # 顶部警报条
        cv2.rectangle(image, (0, 0), (w, 50), (0, 0, 255), -1)
        cv2.putText(image, f"ALARM: {action.upper()}!", (w//4, 35), 
                    cv2.FONT_HERSHEY_DUPLEX, 1, (255, 255, 255), 2)
    else:
        # 绿色正常条
        cv2.rectangle(image, (0, 0), (w, 50), (0, 255, 0), -1)
        cv2.putText(image, f"STATUS: {action.upper()}", (w//3, 35), 
                    cv2.FONT_HERSHEY_DUPLEX, 1, (0, 0, 0), 2)

In [24]:
import collections
import time

# --- 初始化平滑器 ---
prediction_queue = collections.deque(maxlen=20) 
smoothed_action = "attention" 
smoothed_conf = 0.0

cap = cv2.VideoCapture(0)
sequence = []
anomaly_counter = 0
last_alert_time = 0 
ALERT_COOLDOWN = 5 

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break

    image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = holistic.process(image)
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    
    keypoints = extract_keypoints(results)
    sequence.append(keypoints)
    sequence = sequence[-60:] 
    
    if len(sequence) == 60:
        input_data = np.expand_dims(sequence, axis=0).astype(np.float32)
        input_name = ort_sess.get_inputs()[0].name
        res = ort_sess.run(None, {input_name: input_data})[0][0]
        
        # --- 核心改进：逻辑干预 Daydream，给 Attention 机会 ---
        # 拷贝一份结果进行处理，不破坏原始 debug 数据
        modified_res = res.copy()
        
        # 如果 daydream 概率没有达到极其显著的程度 (例如 0.9)，
        # 我们将其权重减半，防止它在 0.7-0.8 波动时霸占 argmax
        if modified_res[2] < 0.9:
            modified_res[2] = modified_res[2] * 0.5
        
        # 使用干预后的结果来选择当前帧动作
        current_idx = np.argmax(modified_res)
        prediction_queue.append(current_idx)

        thresholds = {
            'attention': 0.15,
            'phone': 0.45, 
            'daydream': 0.95, # 配合上面的干预，这里设为 0.95
            'sleepy': 0.7
        }

        counts = collections.Counter(prediction_queue)
        most_common_idx, freq = counts.most_common(1)[0]
        
        if freq > 12:
            candidate_action = actions[most_common_idx]
            candidate_conf = res[most_common_idx] # 使用原始置信度
            
            threshold = thresholds.get(candidate_action, 0.5)
            
            if candidate_action == 'phone' and freq > 15:
                threshold -= 0.15 # 我觉得ok，因为phone的识别还是比较有效的
            
            # 差异化阈值判定
            if candidate_conf > threshold:
                smoothed_action = candidate_action
                smoothed_conf = candidate_conf 
            else:
                # 没过门槛，回归默认状态
                smoothed_action = "attention"
                smoothed_conf = res[0]

        # --- 业务逻辑判断 ---
        is_anomaly = smoothed_action in ['phone', 'sleepy']
        
        if is_anomaly:
            anomaly_counter += 1
        else:
            anomaly_counter = max(0, anomaly_counter - 1)

        if anomaly_counter >= ALERT_LIMIT:
            current_time = time.time()
            if current_time - last_alert_time > ALERT_COOLDOWN:
                print(f"!!! 异常报警: {smoothed_action} !!!")
                try:
                    toaster.show_toast("学习状态异常", f"检测到正在 {smoothed_action}！", duration=2, threaded=True)
                except Exception as e:
                    pass 
                last_alert_time = current_time
            anomaly_counter = 0 
        
        # UI 绘制
        try:
            draw_alert_ui(image, smoothed_action, float(smoothed_conf), is_anomaly)
        except:
            color = (0, 0, 255) if is_anomaly else (0, 255, 0)
            cv2.putText(image, f'{smoothed_action} {float(smoothed_conf):.2f}', (10, 50), 
                        cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)
        
        # Debug 信息显示原始概率 (res)，方便观察
        debug_str = f"Att:{res[0]:.2f} Pho:{res[1]:.2f} Day:{res[2]:.2f} Sle:{res[3]:.2f}"
        cv2.putText(image, debug_str, (10, image.shape[0]-30), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 200, 200), 1)
        vote_str = f"Vote: {actions[most_common_idx]} ({freq}/20)"
        cv2.putText(image, vote_str, (10, image.shape[0]-10), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 0), 1)

    cv2.imshow('Smart Monitor - Stability Mode', image)
    
    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'): break
    elif key == ord('r'): anomaly_counter = 0

cap.release()
cv2.destroyAllWindows()